# 地址到经纬度转换 Notebook

本 Notebook 用于将地址字符串批量转换为地理坐标（纬度 latitude、经度 longitude）。

后续将在此 Notebook 中实现：
- 读取包含地址列的原始数据
- 调用地理编码服务（如 OpenStreetMap Nominatim、Google Maps 等）
- 将返回的经纬度写回数据集并导出


In [2]:
import pandas as pd

# 读取当前目录下的训练集文件
train_path = "ppr-group-25208508-train.csv"
train_df = pd.read_csv(train_path)

# 打印前几行数据
train_df.head()

,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,30/09/2016,"28 BRACKEN COURT, DONNYBROOK, CORK",Cork,NaN,"€181,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,20/12/2016,"2 AN CLOCHAR, CONVENT RD, DONERAILE",Cork,NaN,"€50,152.49",No,Yes,New Dwelling house /Apartment,less than 38 sq metres
2,28/09/2016,"Apartment 7 The Court, Clonattin, Gorey",Wexford,NaN,"€62,171.81",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
3,16/09/2016,"6 Monalin, Wicklow Hills, Newtownmountkennedy",Wicklow,NaN,"€223,348.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
4,29/01/2016,"18 Lislea, Frascati Park, Blackrock",Dublin,NaN,"€310,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


In [5]:
import requests
import urllib.parse

# 取前 10000 行数据进行地理编码示例
sample_df = train_df.head(10000).copy()


def geocode_address(addr: str):
    """调用本地 Nominatim 服务，把地址转换为 "lat,lon" 字符串。
    如果查不到或出错则返回 None。
    """
    if pd.isna(addr):
        return None
    addr_str = str(addr).strip()
    if not addr_str:
        return None

    query = urllib.parse.quote(addr_str)
    url = f"http://localhost:8080/search?q={query}&countrycodes=ie"

    try:
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        if not data:
            return None
        first = data[0]
        lat = first.get("lat")
        lon = first.get("lon")
        if lat is None or lon is None:
            return None
        # 按要求写回到原本的 address 列，这里用 "lat,lon" 形式
        return f"{lat},{lon}"
    except Exception as e:
        print(f"Geocoding error for '{addr_str}': {e}")
        return None


# 对 sample_df 的 Address 列做转换，结果存入 coordinate 列
sample_df["coordinate"] = sample_df["Address"].apply(geocode_address)

# 保存到新文件（前 10000 行）
output_path = "ppr-group-25208508-train-geocoded-top10000.csv"
sample_df.to_csv(output_path, index=False)

sample_df.head()

,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,coordinate
0,30/09/2016,"28 BRACKEN COURT, DONNYBROOK, CORK",Cork,NaN,"€181,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN,"51.8614499,-8.4342193"
1,20/12/2016,"2 AN CLOCHAR, CONVENT RD, DONERAILE",Cork,NaN,"€50,152.49",No,Yes,New Dwelling house /Apartment,less than 38 sq metres,NaN
2,28/09/2016,"Apartment 7 The Court, Clonattin, Gorey",Wexford,NaN,"€62,171.81",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,NaN
3,16/09/2016,"6 Monalin, Wicklow Hills, Newtownmountkennedy",Wicklow,NaN,"€223,348.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,"53.0886448,-6.1134018"
4,29/01/2016,"18 Lislea, Frascati Park, Blackrock",Dublin,NaN,"€310,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN,NaN


In [6]:
# 统计 coordinate 列的有效值和空值数量及百分比
valid_count = sample_df["coordinate"].notna().sum()
null_count = sample_df["coordinate"].isna().sum()

total = len(sample_df)
valid_pct = valid_count / total * 100 if total > 0 else 0
null_pct = null_count / total * 100 if total > 0 else 0

print(f"coordinate 非空数量: {valid_count} ({valid_pct:.2f}%)")
print(f"coordinate 空值数量: {null_count} ({null_pct:.2f}%)")

coordinate 非空数量: 3015 (30.15%)
coordinate 空值数量: 6985 (69.85%)


In [9]:
# 按 Address 中逗号数量分类并统计数量及百分比
addr_series_all = train_df["Address"].fillna("").astype(str)

comma_counts = addr_series_all.apply(lambda x: x.count(","))

summary = (
    comma_counts.value_counts()
    .sort_index()
    .rename_axis("comma_count")
    .reset_index(name="count")
)

total_addr = len(addr_series_all)
summary["percent"] = summary["count"] / total_addr * 100

summary

,comma_count,count,percent
0,1,2757,5.105556
1,2,51243,94.894444


In [10]:
from collections import Counter

# 只使用含有两个逗号（即 3 个片段）的 Address
addr_series = train_df["Address"].dropna().astype(str)
addr_2comma = addr_series[addr_series.apply(lambda x: x.count(",")) == 2]

pos0_counter = Counter()  # 第 1 段
pos1_counter = Counter()  # 第 2 段
pos2_counter = Counter()  # 第 3 段

for addr in addr_2comma:
    parts = [p.strip() for p in addr.split(",")]
    # 理论上有两个逗号就会有 3 段，这里再做一次保护
    if len(parts) >= 3:
        if parts[0]:
            pos0_counter[parts[0]] += 1
        if parts[1]:
            pos1_counter[parts[1]] += 1
        if parts[2]:
            pos2_counter[parts[2]] += 1


def counter_to_df(counter, colname):
    total = sum(counter.values())
    df = (
        pd.DataFrame(counter.items(), columns=[colname, "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    df["percent"] = df["count"] / total * 100 if total > 0 else 0
    return df


seg0_freq = counter_to_df(pos0_counter, "segment_0")
seg1_freq = counter_to_df(pos1_counter, "segment_1")
seg2_freq = counter_to_df(pos2_counter, "segment_2")

# 分别查看每个片段位置前 20 个高频词及其频率
seg0_freq.head(20), seg1_freq.head(20), seg2_freq.head(20)

(       segment_0  count   percent
 0          APT 1    140  0.273208
 1          APT 2     85  0.165876
 2        MAIN ST     72  0.140507
 3    MAIN STREET     72  0.140507
 4    APARTMENT 1     56  0.109283
 5          APT 3     55  0.107332
 6    THE COTTAGE     54  0.105380
 7          APT 5     49  0.095623
 8          APT 6     43  0.083914
 9          APT 4     42  0.081962
 10  THE BUNGALOW     35  0.068302
 11        APT 10     30  0.058545
 12         APT 8     28  0.054642
 13        APT 12     24  0.046836
 14        APT 14     23  0.044884
 15       NEWTOWN     23  0.044884
 16         APT 7     22  0.042933
 17         APT 9     20  0.039030
 18    THE SQUARE     19  0.037078
 19   APARTMENT 4     19  0.037078,
       segment_1  count   percent
 0     BLACKROCK    220  0.429327
 1    CLONDALKIN    218  0.425424
 2         LUCAN    205  0.400055
 3        SWORDS    192  0.374685
 4       DUNDALK    165  0.321995
 5   LETTERKENNY    158  0.308335
 6      DROGHEDA    150  0

In [13]:
from collections import Counter

# 基于“有两个逗号的地址”的第三个片段做细化统计
addr_series = train_df["Address"].dropna().astype(str)
addr_2comma = addr_series[addr_series.apply(lambda x: x.count(",")) == 2]

third_counter = Counter()

for addr in addr_2comma:
    parts = [p.strip() for p in addr.split(",")]
    if len(parts) < 3:
        continue
    seg3 = parts[2].strip()
    # 为了统计时不区分大小写，这里统一转为大写
    seg3 = seg3.upper()
    if not seg3:
        continue

    # 按空格分隔第三个片段
    bits = seg3.split()

    cleaned = seg3  # 默认不变
    if len(bits) == 2:
        b1, b2 = bits
        # 如果其中一侧是纯数字，则舍弃数字、保留另一侧
        if b1.isdigit() and not b2.isdigit():
            cleaned = b2
        elif b2.isdigit() and not b1.isdigit():
            cleaned = b1
    else:
        # 对多于 2 段的情况：去掉纯数字 token，如果全是数字就保留原串
        non_digit_bits = [b for b in bits if not b.isdigit()]
        if non_digit_bits:
            cleaned = " ".join(non_digit_bits)

    # 再次按空格分隔，如果形如 "CO XXX"，则去掉前面的 CO
    if cleaned:
        words = cleaned.split()
        if len(words) >= 2 and words[0].upper() == "CO":
            cleaned = " ".join(words[1:])

    if cleaned:
        third_counter[cleaned] += 1

# 统计第三个片段（清洗后）的频次和百分比
third_freq = (
    pd.DataFrame(third_counter.items(), columns=["segment_2_clean", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

total_third = third_freq["count"].sum()
third_freq["percent"] = third_freq["count"] / total_third * 100 if total_third > 0 else 0

third_freq.head(50)

,segment_2_clean,count,percent
0,DUBLIN,8491,16.570068
1,CORK,2354,4.593798
2,GALWAY,1366,2.665730
3,LIMERICK,1228,2.396425
4,KILDARE,978,1.908553
5,MEATH,767,1.496790
6,WEXFORD,756,1.475323
7,WATERFORD,738,1.440197
8,WICKLOW,710,1.385555
9,TIPPERARY,657,1.282126


In [15]:
# 统计第一个逗号之前包含 "APT"（不区分大小写）的地址数量及百分比
addr_series = train_df["Address"].dropna().astype(str)

total_addr = len(addr_series)

count_apt_first_seg = 0
for addr in addr_series:
    # 只取第一个逗号前的片段
    first_seg = addr.split(",", 1)[0].strip().upper()
    # 含有 APT 或 APARTMENT 都算
    if "APT" in first_seg or "APARTMENT" in first_seg:
        count_apt_first_seg += 1

percent_apt_first_seg = count_apt_first_seg / total_addr * 100 if total_addr > 0 else 0

print(f"总地址数: {total_addr}")
print(f"第一个逗号前包含 'APT' 的地址数: {count_apt_first_seg} ({percent_apt_first_seg:.2f}%)")

总地址数: 54000
第一个逗号前包含 'APT' 的地址数: 3128 (5.79%)
